# Trajectory Risk Analysis: Scoring Risk Signals in Agent Behavior

Based on: [TRACER: Trajectory Risk Aggregation for Critical Episodes](https://arxiv.org/abs/2602.11409) (Feb 2026)

## The Problem

Trajectory scoring tells you *if* the path was good. But for production monitoring, you need to know *how risky* a trajectory is. Specific risk signals include:

| Risk Signal | What It Means | Why It Matters |
|------------|--------------|----------------|
| **Tool failures** | A tool call returned an error | Agent may produce incomplete answer |
| **Duplicate calls** | Same tool called multiple times | Token waste, possible reasoning loop |
| **Irrelevant tools** | Tools called that don't match the query | Confusion, token waste |
| **High call count** | Too many tool calls for a straightforward query | Possible reasoning loop |
| **Long duration** | Individual tool calls taking too long | Latency for end user |

## What We Compare

We run 3 queries through a live agent and compute a **risk score** from the captured trajectory. No LLM needed for risk scoring; it is purely deterministic (instant, zero cost).

In [ ]:
# %pip install strands-agents strands-agents-evals boto3

## Step 1: Define the risk scorer

**What this does:** Defines a deterministic function that computes a risk score (0.0 to 1.0) from a trajectory, based on three weighted factors.

**Why these weights matter:** The risk score is a weighted average of three factors, each contributing differently based on their severity:

| Factor | Weight | What It Measures | Why This Weight |
|--------|:------:|-----------------|-----------------|
| `failure_rate` | **0.4** (highest) | Proportion of tool calls that returned errors | Failures directly impact answer quality — a failed tool means missing information |
| `duplicate_rate` | **0.3** | Proportion of redundant tool calls | Duplicates waste tokens and latency but do not corrupt the answer |
| `excess_calls_rate` | **0.3** | How many calls beyond the expected maximum | Excess calls suggest reasoning loops or confusion, but may sometimes be legitimate |

The formula: `risk = (failure_rate * 0.4) + (duplicate_rate * 0.3) + (excess_rate * 0.3)`

A risk score below 0.2 is "Low" (green), 0.2-0.5 is "Medium" (yellow), and above 0.5 is "High" (red).

> **What to look for:** The low-risk trajectory (2 successful calls) should score near 0.0. The high-risk trajectory (5 calls with 1 failure, duplicates, and excess) should score well above 0.3. Compare the individual factor breakdowns to understand which signals contribute most.

In [ ]:
import nest_asyncio
nest_asyncio.apply()  # Fix for Jupyter async event loop

from collections import Counter


def compute_risk_score(trajectory: list[dict], max_expected_calls: int = 3) -> dict:
    """Compute a risk score from a trajectory. Deterministic, zero LLM cost.

    Risk factors (each 0.0-1.0, weighted equally):
    - failure_rate: proportion of failed tool calls
    - duplicate_rate: proportion of duplicate tool calls
    - excess_calls: how many calls beyond expected maximum
    """
    if not trajectory:
        return {"risk_score": 0.0, "factors": {}}

    total = len(trajectory)
    tool_names = [t["tool"] for t in trajectory]
    counts = Counter(tool_names)

    # Factor 1: failures
    failures = sum(1 for t in trajectory if t.get("had_error") or t.get("status") == "error")
    failure_rate = failures / total

    # Factor 2: duplicates
    duplicates = sum(count - 1 for count in counts.values() if count > 1)
    duplicate_rate = duplicates / total

    # Factor 3: excess calls beyond expected
    excess = max(0, total - max_expected_calls) / max_expected_calls
    excess_rate = min(excess, 1.0)

    # Weighted average
    risk_score = (failure_rate * 0.4) + (duplicate_rate * 0.3) + (excess_rate * 0.3)

    return {
        "risk_score": round(risk_score, 3),
        "risk_level": "🟢 Low" if risk_score < 0.2 else "🟡 Medium" if risk_score < 0.5 else "🔴 High",
        "factors": {
            "failure_rate": round(failure_rate, 3),
            "duplicate_rate": round(duplicate_rate, 3),
            "excess_calls_rate": round(excess_rate, 3),
        },
        "details": {
            "total_calls": total,
            "failures": failures,
            "duplicates": duplicates,
            "unique_tools": list(counts.keys()),
        },
    }


# Quick test with simulated data
low_risk = [{"tool": "search_flights", "status": "success", "had_error": False},
            {"tool": "get_weather", "status": "success", "had_error": False}]

high_risk = [{"tool": "search_flights", "status": "success", "had_error": False},
             {"tool": "search_flights", "status": "error", "had_error": True},
             {"tool": "search_flights", "status": "success", "had_error": False},
             {"tool": "get_currency_exchange", "status": "success", "had_error": False},
             {"tool": "get_weather", "status": "success", "had_error": False}]

print("Low-risk trajectory:", compute_risk_score(low_risk))
print("High-risk trajectory:", compute_risk_score(high_risk))

---
## Step 2: Run live agent and compute risk

**What this does:** Sends 3 queries of increasing complexity through a live agent, captures trajectories with `TrajectoryPlugin`, and computes risk scores for each.

**Why 3 different complexities:** Simple queries (1 tool needed) should produce low-risk trajectories. Complex queries (4 tools needed) may produce higher risk because the agent has more opportunities to make unnecessary calls, duplicate calls, or encounter failures.

> **What to look for in "simple" (weather only):** Expect 1 tool call, risk near 0.0. If the agent calls extra tools for a simple weather query, that is a red flag.

> **What to look for in "multi_tool" (flights + weather):** Expect 2 tool calls, low risk. Watch whether the agent adds unnecessary calls like currency exchange.

> **What to look for in "complex" (flights + weather + currency + hotel):** Expect 4 tool calls. The `max_expected_calls=3` setting means even a perfect execution will show some excess, pushing the risk score up. This is intentional — it demonstrates that the threshold should be calibrated per query type.

In [ ]:
import sys
sys.path.insert(0, "../01-trajectory-scoring")
from trajectory_plugin import TrajectoryPlugin, search_flights, get_weather, book_hotel, get_currency_exchange
from strands import Agent
from strands.models.openai import OpenAIModel

MODEL = "gpt-4o-mini"

QUERIES = [
    ("simple", "What's the weather in Paris?"),
    ("multi_tool", "Find flights NYC to London and check weather there"),
    ("complex", "Find flights NYC to London, check weather, convert 500 USD to GBP, and book Hotel Marriott for March 20-22"),
]

print("=" * 70)
print("LIVE AGENT RISK ANALYSIS")
print("=" * 70)

results = []
for label, query in QUERIES:
    tracker = TrajectoryPlugin()
    agent = Agent(
        model=OpenAIModel(model_id=MODEL),
        tools=[search_flights, get_weather, book_hotel, get_currency_exchange],
        hooks=[tracker],
        system_prompt="You are a travel assistant. Use tools to answer questions.",
    )

    agent(query)

    # Compute risk from captured trajectory
    risk = compute_risk_score(tracker.trajectory, max_expected_calls=3)
    results.append({"label": label, "query": query, "risk": risk, "trajectory": tracker.tool_names})

    print(f"\n  {risk['risk_level']}  {label}")
    print(f"     Query: {query[:50]}")
    print(f"     Tools: {tracker.tool_names}")
    print(f"     Risk score: {risk['risk_score']:.3f}")
    print(f"     Factors: {risk['factors']}")

---
## Comparison: Risk Score vs LLM Judge

**What this does:** Displays the risk scores side-by-side and compares the deterministic risk scorer to the LLM-based `TrajectoryEvaluator`.

**Why compare both:** The deterministic risk score is free and instant, making it ideal for real-time monitoring and alerts. The LLM judge costs tokens and takes 2-5 seconds, but it can assess *semantic* quality — for example, whether a tool call was relevant to the query, which requires understanding the query intent. Use both together: risk scores for monitoring dashboards, LLM evaluation for periodic quality audits.

> **What to look for:** The comparison table shows when each approach excels. The deterministic scorer catches structural issues (failures, duplicates) but cannot judge whether a tool was semantically relevant. The LLM judge catches semantic issues but costs more. The recommendation is to use the risk scorer for every request and the LLM judge on a sample.

In [ ]:
print("=" * 70)
print("COMPARISON: Deterministic Risk vs LLM Trajectory Judge")
print("=" * 70)

print(f"\n  {'Query':<20} {'Risk Score':<15} {'Risk Level':<15} {'Tools Called'}")
print(f"  {'-'*20} {'-'*15} {'-'*15} {'-'*30}")

for r in results:
    risk = r["risk"]
    print(f"  {r['label']:<20} {risk['risk_score']:<15.3f} {risk['risk_level']:<15} {r['trajectory']}")

print(f"""
  ┌─────────────────────────────────────────────────────────────┐
  │ Deterministic Risk Score         │ LLM TrajectoryEvaluator  │
  ├──────────────────────────────────┼──────────────────────────┤
  │ ✅ Instant (0ms)                 │ ⏱️  Slow (2-5s per eval)  │
  │ ✅ Zero cost                     │ 💰 1 LLM call per eval   │
  │ ✅ Catches failures, duplicates  │ ✅ Catches semantic issues│
  │ ❌ Cannot judge relevance        │ ✅ Judges tool relevance  │
  │ Best for: monitoring, alerts     │ Best for: quality gates   │
  └──────────────────────────────────┴──────────────────────────┘

  💡 Use BOTH: Risk score for real-time monitoring (free),
     TrajectoryEvaluator for periodic quality checks (thorough).
""")